# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and inspect the dataset contents using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load Dataset metadata and instantiate dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

We will list all available record sets in the dataset and display their fields and columns using their `@id`s. This helps you choose what to analyze or extract.

In [ ]:
# List all available record sets and their fields/columns
print("Available record sets in the dataset:")
record_set_ids = []
if metadata.record_sets:
    for rs in metadata.record_sets:
        print(f"- RecordSet: @id = {rs.id}, name = {rs.name}")
        record_set_ids.append(rs.id)
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field: @id = {field.id}, name = {field.name}, dataType = {field.data_type}")
        print("  Columns:")
        for col in rs.columns:
            print(f"    - Column: @id = {col.id}, header = {col.header}")
else:
    print("No record sets found in the dataset metadata.")

## 3. Data Extraction
Let's extract data from a specific record set (e.g., the main protein table) into a DataFrame for analysis. 

You should choose the `@id` of the main record set relevant for quantitative analysis (below we attempt extraction for all available record sets).

In [ ]:
# Attempt to extract data for all record sets present
dataframes = {}

if record_set_ids:
    for rs_id in record_set_ids:
        try:
            # The generator yields dict records following the field @id mapping
            print(f"\nExtracting records for RecordSet @id: {rs_id}")
            records = list(dataset.records(record_set=rs_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[rs_id] = df
                print(f"Loaded DataFrame shape: {df.shape}")
                print(f"Columns: {df.columns.tolist()}")
                display(df.head())
            else:
                print("No records found.")
        except Exception as e:
            print(f"Error extracting record set {rs_id}: {e}")
else:
    print("No record sets available to extract records.")

## 4. Exploratory Data Analysis (EDA)
Now we'll perform example EDA operations on the largest available record set. We'll demonstrate filtering based on a numeric field, normalizing it, and (if applicable) grouping by a categorical attribute.

**You may wish to change numeric_field or group_field depending on the exact fields present in the main record set.**

In [ ]:
# Choose main record set for EDA (if more than one, select the largest)
if dataframes:
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"Main record set chosen for EDA: {main_record_set_id}")
    df = dataframes[main_record_set_id]

    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    print(f"Numeric fields available: {numeric_candidates}")
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Use the first detected numeric column
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where field '{numeric_field}' > mean ({threshold}):\n{filtered_df.head()}")

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a likely categorical field
        group_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping filtered data by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Let's visualize the distribution of the main numeric field in the primary record set. A histogram provides a quick overview.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()
else:
    print("No numeric data to plot.")

## 6. Conclusion
We explored metadata, record sets, and fields of the FAIR^2 mass spectrometry dataset, demonstrated extraction with `mlcroissant`, performed a simple exploratory analysis, and visualized a numeric field.

For more detailed analysis, select fields using their `@id` and consult the Croissant schema for domain-specific structure and intended use. The provenence and reproducibility offered by Croissant and FAIR principles make this dataset powerful for both explorative and production-ready workflows.